# Part 2 — deterministic T4 baseline audit

Train exactly **Part-1-config EEGNet** and the **unchanged vanilla transformer**. No custom model is trained.

1. Upload `eeg_dl_exam_3hr` to Google Drive, including Part 2's visible `dataset.zip`.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Set `DRIVE_PROJECT` below, then run all cells.

Defaults: subjects 1–8 / 9–10, 40 epochs, seeds 0/1/2, effective batch 64, FP16 training, strict deterministic kernels. The first seed of each model is repeated and must reproduce exact weights, predictions and loss histories. Six scientific runs plus two verification runs.

The notebook copies data to the runtime's local disk and saves completed runs to Drive. See `COLAB.md` for protocol details and `BASELINE_AUDIT.md` for the architecture walkthrough. T4 runtime will be measured, not assumed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess
from datetime import datetime, timezone

# EDIT THIS PATH if you put the uploaded folder somewhere else.
DRIVE_PROJECT = Path('/content/drive/MyDrive/eeg_dl_exam_3hr')
DRIVE_PART2 = DRIVE_PROJECT / 'part2_transformer'
if not (DRIVE_PART2 / 'colab_audit.py').is_file():
    raise FileNotFoundError(f'Set DRIVE_PROJECT to the folder containing part2_transformer. Looked in {DRIVE_PART2}')

# Use a fresh local working copy; do not copy .venv, old outputs or Part 1.
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
LOCAL_PART2 = Path('/content') / f'basil_part2_{stamp}'
LOCAL_PART2.mkdir()
for pattern in ['*.py', '*.md', 'requirements*.txt']:
    for source in DRIVE_PART2.glob(pattern):
        shutil.copy2(source, LOCAL_PART2 / source.name)
if (DRIVE_PART2 / 'dataset.zip').is_file():
    shutil.copy2(DRIVE_PART2 / 'dataset.zip', LOCAL_PART2 / 'dataset.zip')
    print('Copied dataset.zip. Checksums will be verified before training.')
elif (DRIVE_PART2 / '.cache/mne').is_dir():
    shutil.copytree(DRIVE_PART2 / '.cache/mne', LOCAL_PART2 / '.cache/mne')
else:
    print('No bundled recordings found; prepare_data.py will download them from PhysioNet.')

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
DRIVE_OUTPUTS = DRIVE_PART2 / 'artifacts/colab_gpu'
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Local work:', LOCAL_PART2)
print('Drive results:', DRIVE_OUTPUTS)


## Dependencies and GPU
Keep Colab's preinstalled CUDA-enabled PyTorch. Only install the scientific dependencies; the repository's laptop `.venv` is not used.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LOCAL_PART2 / 'requirements-colab.txt')], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then reconnect and run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Capability:', torch.cuda.get_device_capability(0))
subprocess.run(['nvidia-smi'], check=True)


## Prepare and verify data
Extract the bundled recordings locally, check all 30 SHA-256 hashes against MNE's registry, filter/epoch, and verify the subject split and shapes. The loader preserves the starter's preprocessing.

In [ ]:
subprocess.run([sys.executable, '-B', '-u', 'prepare_data.py'], cwd=LOCAL_PART2, check=True)
import json
print(json.dumps(json.loads((LOCAL_PART2 / 'artifacts/data_summary.json').read_text()), indent=2))


## Train and audit
Settings below define the experiment **before** training. EEGNet uses full batches of 64 to retain its batch-normalization behavior; the transformer accumulates 16-trial microbatches into the same effective batch size. Both use the prespecified final epoch and FP32 final evaluation.

Strict determinism stays enabled. The first seed for each model is independently repeated and checked exactly. Repeat runs are excluded from the accuracy summaries. Results are backed up to Drive after each completed seed. No test-accuracy checkpoint selection or hyperparameter search is performed.

In [ ]:
EPOCHS = 40
SEEDS = [0, 1, 2]
EFFECTIVE_BATCH = 64
TRANSFORMER_MICROBATCH = 16
PRECISION = 'fp16'  # T4: mixed precision with gradient scaling
VERIFY_REPEATS = True

command = [sys.executable, '-B', '-u', 'colab_audit.py',
           '--device', 'cuda', '--precision', PRECISION,
           '--epochs', str(EPOCHS), '--seeds', *map(str, SEEDS),
           '--batch', str(EFFECTIVE_BATCH), '--microbatch', str(TRANSFORMER_MICROBATCH),
           '--strict-determinism', '--sync-dir', str(DRIVE_OUTPUTS)]
if not VERIFY_REPEATS:
    command += ['--no-verify-repeat']

# Stream progress and keep a persistent launcher log.
import signal
log_path = LOCAL_PART2 / 'artifacts/colab_launcher.log'
process = None
try:
    with log_path.open('w') as logfile:
        process = subprocess.Popen(command, cwd=LOCAL_PART2, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line)
            logfile.flush()
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Audit failed with exit code {returncode}. Read the log above and the saved metadata.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=60)
        except subprocess.TimeoutExpired:
            process.terminate()
    raise
finally:
    if log_path.exists():
        shutil.copy2(log_path, DRIVE_OUTPUTS / f'launcher_{stamp}.log')


## Read and export the audit
The report separates training time from test inference, shows every seed and subject, explains both architectures, and records the exact reproducibility checks. Exact repeats apply to this hardware/software stack; future Colab versions can change results.

In [ ]:
from IPython.display import display, Markdown, Image
runs = sorted((LOCAL_PART2 / 'artifacts').glob('colab_*'))
completed = [p for p in runs if p.is_dir() and (p / 'metadata.json').is_file()
             and json.loads((p / 'metadata.json').read_text()).get('status') == 'complete']
if not completed:
    raise RuntimeError('No completed audit. Check the training cell and its saved log.')
RUN_DIR = completed[-1]
display(Markdown((RUN_DIR / 'AUDIT_REPORT.md').read_text()))
display(Image(filename=str(RUN_DIR / 'audit_dashboard.png')))

archive = Path(shutil.make_archive(str(LOCAL_PART2 / RUN_DIR.name), 'zip', root_dir=RUN_DIR))
DRIVE_EXPORT = DRIVE_OUTPUTS / archive.name
shutil.copy2(archive, DRIVE_EXPORT)
print('Detailed run folder:', DRIVE_OUTPUTS / RUN_DIR.name)
print('Export ZIP:', DRIVE_EXPORT)


Optional: download the export ZIP to your laptop after training.

In [ ]:
DOWNLOAD_TO_LAPTOP = False
if DOWNLOAD_TO_LAPTOP:
    from google.colab import files
    files.download(str(archive))
